In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cài đặt giao diện
sns.set_theme(style="whitegrid")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. KHAI BÁO LẠI CÁC CẤU TRÚC (BẮT BUỘC)
# ==========================================
class HierarchicalDataset(Dataset):
    def __init__(self, X, yt, ys, yd):
        self.X = torch.tensor(X, dtype=torch.long)
        self.yt = torch.tensor(yt, dtype=torch.float32)
        self.ys = torch.tensor(ys, dtype=torch.long)
        self.yd = torch.tensor(yd, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yt[i], self.ys[i], self.yd[i]

def build_dataloader(csv_path, vocab, batch_size=512, shuffle=False):
    df = pd.read_csv(csv_path).dropna(subset=['text'])
    df['label_toxic'] = df['tier1_labels'].map({'Clean': 0, 'Toxic': 1}).fillna(0).astype(int)
    df['label_sentiment'] = df['tier2_labels'].map({'Negative': 0, 'Neutral': 1, 'Positive': 2}).fillna(0).astype(int)

    def encode_toxic(label):
        if pd.isna(label) or label in ['Negative', 'Neutral', 'Positive']: return [0, 0, 0]
        return [
            1 if 'Harassment' in str(label) else 0,
            1 if 'Obscene' in str(label) else 0,
            1 if 'Hate Speech' in str(label) else 0 # Chữ S hoa chuẩn xác
        ]
    df['label_toxic_details'] = df['tier2_labels'].apply(encode_toxic)

    X_seq = np.array(df['text'].apply(lambda x: ([vocab.get(w, vocab["<UNK>"]) for w in str(x).split()][:50] + [vocab["<PAD>"]] * 50)[:50]).tolist())
    return DataLoader(HierarchicalDataset(X_seq, df['label_toxic'].values, df['label_sentiment'].values, np.array(df['label_toxic_details'].tolist())), batch_size=batch_size, shuffle=shuffle)

class C_LSTM_Model(nn.Module):
    def __init__(self, vocab_size, w2v_path):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 300, padding_idx=0)
        # Nạp ma trận Word2Vec
        self.emb.weight.data.copy_(torch.from_numpy(np.load(w2v_path)))
        self.lstm = nn.LSTM(300, 128, batch_first=True, bidirectional=True)
        self.convs = nn.ModuleList([nn.Conv1d(256, 100, k) for k in [3,4,5]])
        self.drop = nn.Dropout(0.5)
        self.h_tox = nn.Linear(300, 1)
        self.h_sent = nn.Linear(300, 3)
        self.h_tdet = nn.Linear(300, 3)

    def forward(self, x):
        lstm_out, _ = self.lstm(self.emb(x))
        cnn_in = lstm_out.permute(0, 2, 1)
        feat = self.drop(torch.cat([F.max_pool1d(F.relu(c(cnn_in)), c(cnn_in).shape[2]).squeeze(2) for c in self.convs], 1))
        return self.h_tox(feat).squeeze(1), self.h_sent(feat), self.h_tdet(feat)

# ==========================================
# 2. HÀM ĐÁNH GIÁ TỔNG THỂ VÀ VẼ BIỂU ĐỒ
# ==========================================
def evaluate_and_plot_overall(model, dataloader, phase_name="Test Data"):
    model.eval()

    total_samples, exact_matches = 0, 0
    t1_correct, t2a_correct, t2a_total, t2b_exact_matches, t2b_total = 0, 0, 0, 0, 0

    print(f"⏳ Đang quét {phase_name} để tính điểm Exact Match...")
    with torch.no_grad():
        for x, yt, ys, yd in dataloader:
            x = x.to(DEVICE)
            l_tox, l_sent, l_tdet = model(x)

            # --- Lấy dự đoán ---
            pred_t1 = (torch.sigmoid(l_tox) >= 0.5).int().cpu().numpy()
            pred_t2a = torch.argmax(torch.softmax(l_sent, dim=1), dim=1).cpu().numpy()
            pred_t2b = (torch.sigmoid(l_tdet) >= 0.5).int().cpu().numpy()

            # --- Lấy nhãn thật ---
            true_t1 = yt.int().numpy()
            true_t2a = ys.numpy()
            true_t2b = yd.int().numpy()

            batch_size = len(true_t1)
            total_samples += batch_size

            # --- Logic chấm điểm Khớp tuyệt đối ---
            for i in range(batch_size):
                is_exact_match = False

                if pred_t1[i] == true_t1[i]: t1_correct += 1

                if true_t1[i] == 0:  # Nhãn gốc CLEAN
                    t2a_total += 1
                    if pred_t2a[i] == true_t2a[i]:
                        t2a_correct += 1
                        if pred_t1[i] == 0: is_exact_match = True

                elif true_t1[i] == 1:  # Nhãn gốc TOXIC
                    t2b_total += 1
                    if np.array_equal(pred_t2b[i], true_t2b[i]):
                        t2b_exact_matches += 1
                        if pred_t1[i] == 1: is_exact_match = True

                if is_exact_match: exact_matches += 1

    # Tính phần trăm
    overall_exact_acc = (exact_matches / total_samples) * 100
    t1_acc = (t1_correct / total_samples) * 100
    t2a_acc = (t2a_correct / t2a_total) * 100 if t2a_total > 0 else 0
    t2b_acc = (t2b_exact_matches / t2b_total) * 100 if t2b_total > 0 else 0

    # --- VẼ BIỂU ĐỒ ---
    metrics = ['Tier 1\n(Phân loại)', 'Tier 2A\n(Cảm xúc)', 'Tier 2B\n(Chi tiết)', 'Exact Match\n(Toàn hệ thống)']
    scores = [t1_acc, t2a_acc, t2b_acc, overall_exact_acc]
    colors = ['#4c72b0', '#55a868', '#dd8452', '#c44e52']

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(metrics, scores, color=colors, width=0.6)

    ax.set_title(f'Biểu đồ 3: Đánh giá Độ chính xác Tổng thể ({phase_name})', fontsize=16, pad=20)
    ax.set_ylabel('Độ chính xác (Accuracy %)', fontsize=12)
    ax.set_ylim(0, 110)

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('overall_exact_match_performance.png', dpi=300)
    plt.show()

    print("\n" + "★"*50)
    print(f"🏆 BÁO CÁO TỔNG THỂ HỆ THỐNG ({phase_name.upper()})")
    print(f"Tổng số câu đã duyệt        : {total_samples:,}")
    print(f"Số câu đoán khớp tuyệt đối  : {exact_matches:,}")
    print(f"Exact Match Accuracy        : {overall_exact_acc:.2f}%")
    print("★"*50)

# ==========================================
# 3. THỰC THI LOAD MÔ HÌNH VÀ CHẤM ĐIỂM
# ==========================================
if __name__ == "__main__":
    # Đường dẫn file đã lưu
    TEST_CSV_PATH = "test_data.csv"
    VOCAB_PATH = "vocab.json"
    W2V_PATH = "custom_word2vec.npy"
    MODEL_PATH = "best_c_lstm_model.pth"

    if not os.path.exists(MODEL_PATH):
        print("🚨 Lỗi: Không tìm thấy file trọng số mô hình! Vui lòng kiểm tra lại đường dẫn.")
    else:
        print("1. Đang load Từ điển và Dữ liệu Test...")
        with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
            vocab = json.load(f)

        test_loader = build_dataloader(TEST_CSV_PATH, vocab, batch_size=512, shuffle=False)

        print("2. Đang nạp Trọng số (Bộ não) tốt nhất của C-LSTM...")
        model = C_LSTM_Model(len(vocab), W2V_PATH).to(DEVICE)

        # Load map_location=DEVICE để đảm bảo chạy được cả trên CPU/GPU
        checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])

        print("3. Bắt đầu chấm điểm Exact Match...")
        evaluate_and_plot_overall(model, test_loader, phase_name="Test Data")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 BÁO CÁO TỔNG THỂ HỆ THỐNG (TEST DATA)
Tổng số câu đã duyệt        : 81,726
Số câu đoán khớp tuyệt đối  : 50,475
Exact Match Accuracy        : 61.76%
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
